# Módulo 1 — Programación Sincrónica vs Asíncrona en Python (asyncio)
Curso: Python para Ingeniería de Datos

**Objetivo de este notebook:** entender *qué es asincronía*, cuándo conviene, y practicar con ejemplos súper cotidianos (delivery) y ejemplos típicos de Data Engineer (esperas de red/API simuladas con `sleep`).

> Nota: Este notebook está pensado para **Databricks / Colab / Jupyter**.
> En notebooks, se ejecuta `main()` con **`await main()`** (no con `asyncio.run(...)` en la mayoría de casos).


## 1) Sincronía vs Asincronía (teoría corta)
**Sincrónico** = haces una cosa y *no avanzas* hasta que termine.

**Asíncrono** = haces varias cosas que involucran **espera** (I/O: red, disco, APIs) y, mientras una espera, **avanzas con otras**.

### ¿Cuándo sirve de verdad?
- ✅ I/O bound: llamadas a APIs, descargar archivos, leer/escribir a disco, requests a servicios.
- ❌ CPU bound: cálculos pesados (ahí conviene multiprocessing u optimización).


## 2) Conceptos clave de `asyncio`
- **`async def`**: define una *corutina* (una función que puede pausar y reanudar).
- **corutina**: una tarea que se ejecuta dentro de un **event loop** y puede ceder control en puntos `await`.
- **`await`**: *pausa sin bloquear*; devuelve el control al event loop para que ejecute otras tareas mientras se espera.
- **`asyncio.gather(...)`**: ejecuta varias corutinas concurrentemente y espera que todas terminen.
- **event loop**: el “coordinador” que decide qué corutina corre en cada momento.

### ¿Por qué `sleep`?
- `time.sleep(...)` bloquea todo (sincrónico).
- `await asyncio.sleep(...)` simula una espera (red/API) **sin bloquear**, permitiendo que otras tareas avancen.

### Ejecutar `main()`
- En **scripts**: `asyncio.run(main())`
- En **notebooks**: normalmente `await main()` (porque ya existe un event loop).


---
# Parte A — Delivery: ejemplo cotidiano


In [0]:
# ✅ Utilidad: medidor de tiempo (para ver segundos totales)
import time

def medir_tiempo(func):
    """Decorador simple para medir el tiempo de ejecución (sync)."""
    def wrapper(*args, **kwargs):
        inicio = time.time()
        result = func(*args, **kwargs)
        fin = time.time()
        print(f"⏱ Tiempo total: {fin - inicio:.2f} segundos")
        return result
    return wrapper


## A1) Delivery SINCRÓNICO (uno por uno)
**Idea:** pides pizza, esperas que llegue, luego pides hamburguesa, esperas, luego gaseosa, esperas.

Aquí usamos `time.sleep(...)` para simular la espera (como si fuera el tiempo de entrega).


In [0]:
import time

def pedido_sync(nombre: str, segundos: int):
    print(f"[SYNC] Pidiendo {nombre}...")
    time.sleep(segundos)  # BLOQUEA: el programa se detiene aquí y no avanza con nada más
    print(f"[SYNC] Llegó {nombre} ✅")

@medir_tiempo
def demo_delivery_sync():
    pedido_sync("Pizza", 2)
    pedido_sync("Hamburguesa", 3)
    pedido_sync("Gaseosa", 1)

demo_delivery_sync()


## A2) Delivery ASÍNCRONO (todos a la vez)
**Idea:** haces los 3 pedidos de una y vas recibiéndolos cuando llegan.

Aquí usamos:
- `async def` para definir corutinas
- `await` para *esperar sin bloquear*
- `asyncio.gather(...)` para lanzar varias corutinas concurrentemente

> En notebook, ejecuta con `await main()`


In [0]:
import asyncio
import time

async def pedido_async(nombre: str, segundos: int):
    print(f"[ASYNC] Pidiendo {nombre}...")

    # await = "espera sin bloquear"
    # Mientras esta corutina espera, el event loop puede ejecutar otras corutinas.
    await asyncio.sleep(segundos)  # Simula latencia (delivery/red/API)

    print(f"[ASYNC] Llegó {nombre} ✅")
    return nombre

async def main_delivery_async():
    inicio = time.time()

    # gather = ejecutar varias corutinas concurrentemente
    resultados = await asyncio.gather(
        pedido_async("Pizza", 2),
        pedido_async("Hamburguesa", 3),
        pedido_async("Gaseosa", 1),
    )

    fin = time.time()
    print(f"⏱ Tiempo total ASYNC: {fin - inicio:.2f} segundos")
    print("📦 Pedidos recibidos:", resultados)

# ✅ En notebook (Databricks/Colab/Jupyter):
await main_delivery_async()


### ¿Qué deberías observar?
- En SYNC, el tiempo total ≈ 2 + 3 + 1 = **6s**
- En ASYNC, el tiempo total ≈ **3s** (el más lento)

Porque en ASYNC las esperas ocurren al mismo tiempo: cuando una tarea está en `await`, el loop aprovecha y ejecuta otra.


---
# Parte B — Ejemplos sencillos (pero típicos de Data Engineer)


## B1) Simular llamadas a API (sin internet)
En DE, es común tener que enriquecer registros consultando un servicio.

Aquí `asyncio.sleep` simula la latencia de red.


In [0]:
import asyncio
import random
import time

async def fake_api(cliente_id: int):
    # Simula una latencia de red variable (0.3 a 1.2s)
    await asyncio.sleep(random.uniform(0.3, 1.2))
    # Retorna algo como si viniera de una API
    return {"cliente_id": cliente_id, "score": random.randint(300, 900)}

async def main_fake_api():
    inicio = time.time()

    clientes = [101, 102, 103, 104, 105, 106]
    resultados = await asyncio.gather(*(fake_api(c) for c in clientes))

    fin = time.time()
    print(f"⏱ Tiempo total ASYNC (fake API): {fin - inicio:.2f} s")
    print("Resultados:", resultados)

await main_fake_api()


> `resultados = await asyncio.gather(*(fake_api(c) for c in clientes))`  
Esta línea ejecuta **todas las llamadas a la función `fake_api` para cada cliente** al mismo tiempo (de forma asíncrona), espera que todas terminen, y guarda los resultados en la lista `resultados`.  
- `asyncio.gather(...)` lanza varias tareas concurrentemente.
- El `*` desempaqueta la lista de corutinas generadas por el for.
- `await` espera a que todas terminen.

## B2) Comparación rápida: 3 esperas SYNC vs ASYNC
Este es el ejemplo más mínimo para fijar la idea.


In [0]:
import time
import asyncio

# SYNC
inicio = time.time()
time.sleep(1)  # BLOQUEA
time.sleep(1)  # BLOQUEA
time.sleep(1)  # BLOQUEA
print(f"⏱ SYNC (3x1s): {time.time() - inicio:.2f} s")




In [0]:
# ASYNC
async def main_3_esperas():
    inicio = time.time()
    await asyncio.gather(
        asyncio.sleep(1),  # await dentro de gather: esperas concurrentes
        asyncio.sleep(1),
        asyncio.sleep(1),
    )
    print(f"⏱ ASYNC (3x1s): {time.time() - inicio:.2f} s")

await main_3_esperas()

## B3) Manejo de errores sin tumbar todo (`return_exceptions=True`)
En pipelines reales, una parte puede fallar y tú quieres **registrar el error** y seguir con lo demás.


In [0]:
import asyncio

async def tarea(i: int):
    await asyncio.sleep(0.4)
    if i == 3:
        raise ValueError("Falló la tarea 3 (simulado)")
    return f"ok-{i}"

async def main_errores_controlados():
    resultados = await asyncio.gather(
        *(tarea(i) for i in range(1, 6)),
        return_exceptions=True  # evita que un error mate todo el conjunto
    )

    for r in resultados:
        if isinstance(r, Exception):
            print("❌ Error capturado:", r)
        else:
            print("✅ Resultado:", r)

await main_errores_controlados()


---
# Parte C — Ejercicios (para practicar)
Hazlos en clase en parejas o individual. La idea es que escriban **versión sincrónica** y **versión asíncrona**, y comparen tiempos.

💡 Tip: usen `time.time()` para medir.


## Ejercicio 1 — 5 pedidos (SYNC vs ASYNC)
**Tarea:** Simula 5 pedidos con esperas: [1, 2, 1, 3, 2] segundos.
- (a) Hazlo sincrónico usando `time.sleep`.
- (b) Hazlo asíncrono usando `asyncio.gather` + `await asyncio.sleep`.
- (c) Compara tiempos.


In [0]:
# ✅ Tu solución SYNC aquí

import time

def pedido_sync(nombre, segundos):
    print(f"[SYNC] {nombre}...")
    time.sleep(segundos)
    print(f"[SYNC] {nombre} listo ✅")

inicio = time.time()

# TODO: llama 5 pedidos con delays [1, 2, 1, 3, 2]
pedido_sync("Pedido 1", 1)
# ...

print(f"⏱ Tiempo SYNC: {time.time() - inicio:.2f} s")


In [0]:
# ✅ Tu solución ASYNC aquí

import asyncio
import time

async def pedido_async(nombre, segundos):
    print(f"[ASYNC] {nombre}...")
    await asyncio.sleep(segundos)  # espera sin bloquear
    print(f"[ASYNC] {nombre} listo ✅")
    return nombre

async def main():
    inicio = time.time()
    # TODO: usa asyncio.gather con 5 pedidos y delays [1, 2, 1, 3, 2]
    resultados = await asyncio.gather(
         #pedido_async("Pedido 1", 1),
         #.
         #.
         #.
         #.
     )
    fin = time.time()
    print(f"⏱ Tiempo ASYNC: {fin - inicio:.2f} s")
    # print("Resultados:", resultados)

await main()


## Ejercicio 2 — Fake API (SYNC vs ASYNC)
Simula 8 llamadas a API.
- SYNC: usa `time.sleep(random.uniform(0.2, 0.8))`
- ASYNC: usa `await asyncio.sleep(random.uniform(0.2, 0.8))` con `gather`
Mide el tiempo total y compara.


In [0]:
# ✅ SYNC (guía)

import time, random

def fake_api_sync(i):
    time.sleep(random.uniform(0.2, 0.8))  # bloquea
    return {"id": i, "ok": True}

inicio = time.time()

# TODO: llama 8 veces y guarda resultados
# resultados = [fake_api_sync(i) for i in range(8)]

print(f"⏱ Tiempo SYNC: {time.time() - inicio:.2f} s")
# print(resultados)


In [0]:
# ✅ ASYNC (guía)

import asyncio, time, random

async def fake_api_async(i):
    await asyncio.sleep(random.uniform(0.2, 0.8))  # no bloquea
    return {"id": i, "ok": True}

async def main():
    inicio = time.time()
    # TODO: ejecuta 8 llamadas concurrentes
    # resultados = .............
    fin = time.time()
    print(f"⏱ Tiempo ASYNC: {fin - inicio:.2f} s")
    # print(resultados)

await main()


## Ejercicio 3 — Errores controlados
Crea 6 tareas asíncronas donde 2 fallen (por ejemplo i==2 e i==5).
Usa `return_exceptions=True` y muestra cuáles fallaron sin detener el resto.


In [0]:
import asyncio

async def tarea(i):
    await asyncio.sleep(0.3)
    # TODO: haz fallar 2 tareas
    # if i in (...):
    #     raise ValueError(...)
    return f"ok-{i}"

async def main():
    resultados = await asyncio.gather(*(tarea(i) for i in range(1, 7)), return_exceptions=True)
    for r in resultados:
        if isinstance(r, Exception):
            print("❌ Error:", r)
        else:
            print("✅", r)

await main()


---
# Checklist mental (para llevarse)
- Si hay **mucha espera** (API/red/disco), async suele reducir el tiempo total.
- `await` = esperar sin bloquear.
- `gather` = ejecutar muchas esperas a la vez.
- En notebook: `await main()`.
- En script: `asyncio.run(main())`.

✅ Con esto ya puedes explicar asincronía con ejemplos cotidianos + enfoque DE.
